[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C57_Small_Object_Detection_Course/04_slicing/04_slicing_inference.ipynb)

# 04 · 切片推理与高分辨率策略（SAHI 切片 / 重叠率下界 / 跨片 NMM / 两级级联）

目标：把「切片推理」从一个方法名变成一套**能算清代价、能推出参数、能自己实现**的工程方案，
并亲手论证它<b>为什么在车端量产系统里基本不可用</b>、替代方案是什么。

本 notebook 你会亲手实现：
1. **切片网格生成**（尺寸 s / 步长 / 重叠率 r，含贴边补片），以及片数随 r 的**阶梯效应**
2. **坐标还原**（局部 → 全图）与跨片重复检出的量化
3. **重叠率下界 `s·r ≥ d_max` 的数值验证**——穷举所有位置，证明下界是紧的
4. **跨片合并**：IoU / IoS 的差别，NMS（抑制）与 **NMM（合并）**，以及「贴边门控」
5. **代价账**：切片数、像素吞吐的渐近律 `N·s² ≈ W·H/(1-r)²`、车端延迟预算对照
6. **两级级联** vs 切片 vs 全图下采样的**端到端延迟与召回**对比，以及时序累积如何改变结论

> 心智模型：**切片没有创造信息，它只是拒绝丢弃信息。
> 收益 = 你在 resize 阶段丢掉的那部分；代价 = 4–16 倍的算力。**

## 1 · 切片网格：尺寸 s、步长与重叠率 r

先把几何写死。步长 `stride = s(1-r)`，重叠像素 `s·r`。
切片原点取 `0, stride, 2·stride, …`，**最后一片贴右边界**保证覆盖完整（SAHI 的标准做法）。

In [ ]:
import numpy as np, math
from collections import Counter
np.set_printoptions(precision=3, suppress=True)

W, H = 1920, 1080      # 车载前视主摄的典型分辨率
S_SIGN = 0.6           # 限速牌物理直径 60 cm
F_PX = 1662.77         # 60° 水平 FOV / 1920 宽 的等效焦距（像素）—— 模块 05 会完整推导

def slice_origins(total, s, stride):
    '''一维切片起点：0, stride, 2*stride, ...，最后一片贴边界保证覆盖完整。'''
    if total <= s:
        return [0]
    xs = list(range(0, total - s + 1, stride))
    if xs[-1] != total - s:
        xs.append(total - s)          # 贴边补片：代价是它与前一片的重叠更大
    return xs

def make_slices(W, H, s, r):
    '''返回 [(x0,y0,x1,y1), ...] 与步长。r = 重叠率。'''
    stride = max(1, int(round(s * (1.0 - r))))
    xs = slice_origins(W, s, stride)
    ys = slice_origins(H, s, stride)
    return [(x, y, x + s, y + s) for y in ys for x in xs], stride

slices, stride = make_slices(W, H, 640, 0.20)
print(f'W×H = {W}×{H},  s = 640,  r = 0.20  ->  stride = {stride},  overlap = {640-stride} px')
for i, sl in enumerate(slices):
    print(f'  slice {i}: x[{sl[0]:4d},{sl[2]:4d})  y[{sl[1]:4d},{sl[3]:4d})')
assert len(slices) == 8 and stride == 512
print(f'\n切片数 N = {len(slices)}   放大倍数 W/s = {W/640:.1f}×（目标线性尺寸放大 3 倍、面积 9 倍）')

In [ ]:
# ① 覆盖完整性：每个像素至少被一片覆盖（在 1/8 网格上验证）
cov = np.zeros((H // 8, W // 8), dtype=int)
for (x0, y0, x1, y1) in slices:
    cov[y0 // 8:y1 // 8, x0 // 8:x1 // 8] += 1
print('像素被覆盖次数的分布:', dict(zip(*[a.tolist() for a in np.unique(cov, return_counts=True)])))
assert cov.min() >= 1, '存在未被任何切片覆盖的像素'
print(f'最少覆盖 {cov.min()} 次，最多 {cov.max()} 次  ->  重叠带与四角会被重复处理')

# ② 片数是 r 的**阶梯函数** —— 阶梯上有免费午餐
print(f"\n{'r':>6s} {'stride':>7s} {'overlap px':>11s} {'横':>4s} {'纵':>4s} {'N':>4s} {'像素吞吐(Mpx)':>14s}")
for r in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    sl, st = make_slices(W, H, 640, r)
    nx, ny = len(slice_origins(W, 640, st)), len(slice_origins(H, 640, st))
    print(f'{r:>6.2f} {st:>7d} {640-st:>11d} {nx:>4d} {ny:>4d} {len(sl):>4d} {len(sl)*640*640/1e6:>14.2f}')
n10 = len(make_slices(W, H, 640, 0.10)[0])
n30 = len(make_slices(W, H, 640, 0.30)[0])
assert n10 == n30 == 8, (n10, n30)
print('\n✅ r 从 0.10 涨到 0.30，片数都是 8 —— **把 r 从下界推到片数跳变前，是免费的鲁棒性**')
print('   调参顺序：先按「目标要放大到多少像素」定 s，再按下界定 r，最后在阶梯上把 r 往上推')

## 2 · 坐标还原与跨片重复检出

检测器在切片里输出的是**局部坐标**，还原只需加上切片原点。
但重叠带里的目标会被多片各检出一次——这是 NMS 该处理的情形。

In [ ]:
def make_scene(n=12, seed=7):
    '''合成一张街景里的交通标志：像素尺寸由针孔模型 p = f·S/Z 给出（远小近大），
       位置集中在地平线带附近（模块 05 会精确推导这条带），互相保持最小间距。'''
    g = np.random.default_rng(seed)
    recs, tries = [], 0
    while len(recs) < n and tries < 50000:
        tries += 1
        Z = g.uniform(18.0, 150.0)                 # 距离 18–150 m
        p = F_PX * S_SIGN / Z                      # 像素尺寸：55.4 px（18m）… 6.7 px（150m）
        cx = g.uniform(0.06 * W, 0.94 * W)
        cy = H * 0.42 + g.uniform(-90.0, 90.0)     # 地平线带附近
        if any(abs(cx - c) < 130 and abs(cy - d) < 130 for c, d, _, _ in recs):
            continue                               # 保持最小间距，避免合成出「本来就该合并」的歧义
        recs.append((cx, cy, p, Z))
    boxes = np.array([[c - p / 2, d - p / 2, c + p / 2, d + p / 2] for c, d, p, _ in recs])
    zs = np.array([z for _, _, _, z in recs])
    return boxes, zs

GT, GT_Z = make_scene(12, seed=7)
sizes = GT[:, 2] - GT[:, 0]
order = np.argsort(GT_Z)
print(f"{'#':>3s} {'距离 Z(m)':>10s} {'像素尺寸':>9s} {'中心 (cx, cy)':>22s}")
for i in order:
    cx, cy = (GT[i, 0] + GT[i, 2]) / 2, (GT[i, 1] + GT[i, 3]) / 2
    print(f'{i:>3d} {GT_Z[i]:>10.1f} {sizes[i]:>9.1f} {f"({cx:7.1f}, {cy:6.1f})":>22s}')
print(f'\n共 {len(GT)} 个标志：{sizes.min():.1f} – {sizes.max():.1f} px；'
      f'其中 < 32 px（COCO small）的有 {int((sizes < 32).sum())} 个')
assert sizes.min() > F_PX * S_SIGN / 150 - 1e-9 and sizes.max() < F_PX * S_SIGN / 18 + 1e-9
D_MAX = float(sizes.max())
print(f'本图的 d_max = {D_MAX:.1f} px  ->  重叠率下界 r >= d_max/s = {D_MAX/640:.4f}')

In [ ]:
def detect_in_slice(gt, sl, vis_thr=0.9):
    '''模拟检测器在一个切片上的输出：GT 与切片的交集占 GT 面积 >= vis_thr 才检出，
       返回的是**被切片截断后的框**（局部坐标）—— 这正是跨片合并要处理的东西。'''
    x0, y0, x1, y1 = sl
    out = []
    for i, (gx0, gy0, gx1, gy1) in enumerate(gt):
        ix0, iy0 = max(gx0, x0), max(gy0, y0)
        ix1, iy1 = min(gx1, x1), min(gy1, y1)
        iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0)
        vis = iw * ih / ((gx1 - gx0) * (gy1 - gy0))
        if vis >= vis_thr:
            out.append({'box_local': (float(ix0 - x0), float(iy0 - y0),
                                      float(ix1 - x0), float(iy1 - y0)),
                        'score': float(0.50 + 0.45 * vis), 'gt_id': i, 'vis': float(vis)})
    return out

per_slice = [detect_in_slice(GT, sl, vis_thr=0.9) for sl in slices]
print(f"{'切片':>5s} {'检出数':>7s}  命中的 gt_id")
for i, ds in enumerate(per_slice):
    print(f'{i:>5d} {len(ds):>7d}  {[d["gt_id"] for d in ds]}')

# ---- 坐标还原：局部 -> 全图（只是平移；若切片还被 letterbox 过，必须先反 letterbox）----
glob = []
for (x0, y0, _, _), ds in zip(slices, per_slice):
    for d in ds:
        bx0, by0, bx1, by1 = d['box_local']
        glob.append({'box': (bx0 + x0, by0 + y0, bx1 + x0, by1 + y0),
                     'score': d['score'], 'gt_id': d['gt_id']})
cnt = Counter(d['gt_id'] for d in glob)
print(f'\n还原后共 {len(glob)} 个框，对应 {len(cnt)} 个真值目标')
print('被多片重复检出的目标:', {k: v for k, v in sorted(cnt.items()) if v > 1})
assert set(cnt) == set(range(len(GT))), 'r=0.2 满足下界，所有目标都应被某片完整包含'
assert len(glob) > len(cnt), '重叠带必然产生重复检出'
print(f'\n✅ 重复率 = {len(glob)/len(cnt):.2f}×  —— 这些是几乎重合的框，标准 NMS 就能处理')
print('⚠️  下一节会看到：r **低于下界**时产生的是「半截框」，NMS 处理不了')

## 3 · 重叠率的下界：`s·r ≥ d_max`

推导（一维）：目标占 `[x, x+d]`，第 i 片覆盖 `[o_i, o_i+s]`。
完整包含 ⟺ `x+d-s ≤ o_i ≤ x`，即合法原点必须落在一个**长度 s−d 的区间**里。
等差数列（公差 stride）与任意长度 L 的闭区间必然相交 ⟺ `stride ≤ L`。于是

    stride ≤ s − d   ⟺   s(1−r) ≤ s − d   ⟺   **s·r ≥ d**

下面穷举**所有整数位置**验证这个下界是**紧的**。

In [ ]:
def uncovered_positions(total, s, r, d):
    '''穷举所有整数位置，返回「无法被任何一片完整包含」的目标左边界列表 + 切片原点。'''
    stride = max(1, int(round(s * (1.0 - r))))
    origins = slice_origins(total, s, stride)
    bad = [x for x in range(0, total - d + 1)
           if not any(o <= x and x + d <= o + s for o in origins)]
    return bad, origins

s_, d_ = 640, 64
print(f'切片 s = {s_},  目标 d = {d_}   ->   理论下界  r >= d/s = {d_/s_:.4f}')
print(f"\n{'r':>6s} {'stride':>7s} {'overlap px':>11s} {'必然被截断的位置数':>20s} {'':>3s}")
for r in [0.00, 0.02, 0.05, 0.07, 0.09, 0.10, 0.12, 0.20]:
    bad, _ = uncovered_positions(W, s_, r, d_)
    st = max(1, int(round(s_ * (1 - r))))
    print(f'{r:>6.2f} {st:>7d} {s_-st:>11d} {len(bad):>20d} {"✅" if not bad else "❌":>3s}')

b09, _ = uncovered_positions(W, s_, 0.09, d_)
b10, _ = uncovered_positions(W, s_, 0.10, d_)
assert len(b10) == 0, 'r = d/s 恰好达到下界，应当零失败'
assert len(b09) > 0, 'r 比下界小一点点就出现必然截断的位置'
print(f'\nr = 0.09  overlap = {s_-int(round(s_*0.91))} px  <  d = 64  ->  失败位置 {b09}')
print(f'r = 0.10  overlap = {s_-int(round(s_*0.90))} px  =  d = 64  ->  失败位置 {b10}')
print('\n✅ **下界是紧的**：overlap_px >= d 时零失败；少 1 个像素就出现两段必然失败的区间')
print('⚠️  注意失败位置是**固定的几条竖线**（切片边界附近）——')
print('    这类漏检与位置强相关，在随机划分的验证集上被平均掉，上路后表现为「某些路段总漏牌」')

In [ ]:
# ---- r 低于下界时会发生什么：构造一个必然被截断的目标 ----
r_bad = 0.05
sl_bad, st_bad = make_slices(W, H, 640, r_bad)
b_bad, org_bad = uncovered_positions(W, 640, r_bad, 64)
print(f'r = {r_bad}: stride = {st_bad}, overlap = {640-st_bad} px  <  d = 64 px  ❌')
print(f'切片原点 = {org_bad};  必然失败的左边界区间 = [{min(b_bad)}, {max(b_bad)}] 等')

target = np.array([[590.0, 380.0, 654.0, 444.0]])       # 64×64，左边界 590 落在失败区间里
TGT = tuple(float(v) for v in target[0])
print(f'\n目标框 = {TGT}   (左边界 590 ∈ 失败区间)')

strict = [detect_in_slice(target, sl, vis_thr=0.9) for sl in sl_bad]
n_strict = sum(len(x) for x in strict)
print(f'严格检测器（可见率 >= 0.90 才检出）: 全部 {len(sl_bad)} 片共检出 {n_strict} 个  ->  **完全漏检**')
assert n_strict == 0, '低于下界 + 严格检测器 = 跨边界目标被彻底吞掉'

lenient = [detect_in_slice(target, sl, vis_thr=0.5) for sl in sl_bad]
frags = []
for (x0, y0, _, _), ds in zip(sl_bad, lenient):
    for dd in ds:
        bx0, by0, bx1, by1 = dd['box_local']
        frags.append(((bx0 + x0, by0 + y0, bx1 + x0, by1 + y0), dd['score'], dd['vis']))
print(f'\n宽松检测器（可见率 >= 0.50）: 得到 {len(frags)} 个**半截框**')
for b, sc, v in frags:
    print(f'  {tuple(round(t,1) for t in b)}   score={sc:.3f}   可见率={v:.3f}')
assert len(frags) == 2
print('\n⚠️  半截框的 score 仍然不低（模型看到半个红圈也会给高分），但**框是错的、尺寸小了一半**。')
print('    TSR 里若按框的像素尺寸反推距离，半截框会让距离估计直接翻倍出错。')

## 4 · 跨片合并：IoU vs IoS，NMS（抑制）vs NMM（合并）

跨片截断场景要问的不是「两个框有多像」（IoU），
而是「**小的那个有多大比例被包住**」（IoS = Intersection over Smaller area）。

In [ ]:
def iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def ios(a, b):
    '''Intersection over Smaller area：度量「小框有多大比例被大框包住」。'''
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    sm = min((a[2]-a[0])*(a[3]-a[1]), (b[2]-b[0])*(b[3]-b[1]))
    return inter / sm if sm > 0 else 0.0

def nms(boxes, scores, thr=0.6):
    '''标准 NMS：**抑制**（删掉）高 IoU 的低分框。'''
    idx = [int(i) for i in np.argsort(scores)[::-1]]
    keep = []
    while idx:
        i = idx.pop(0)
        keep.append(i)
        idx = [j for j in idx if iou(boxes[i], boxes[j]) <= thr]
    return keep

def nmm(boxes, scores, ios_thr=0.5):
    '''Non-Maximum **Merging**：把 IoS 高的框**并起来**（取包围盒），而不是删掉。'''
    order = [int(i) for i in np.argsort(scores)[::-1]]
    used, out = set(), []
    for i in order:
        if i in used:
            continue
        used.add(i)
        bx, sc, members = list(boxes[i]), float(scores[i]), [i]
        for j in order:
            if j in used:
                continue
            if ios(tuple(bx), boxes[j]) >= ios_thr:
                used.add(j); members.append(j)
                bx = [min(bx[0], boxes[j][0]), min(bx[1], boxes[j][1]),
                      max(bx[2], boxes[j][2]), max(bx[3], boxes[j][3])]
                sc = max(sc, float(scores[j]))
        out.append({'box': tuple(bx), 'score': sc, 'members': members})
    return out

print('IoU / IoS / NMS / NMM 就位')
# 自检：一个小框完全落在大框内部
small, big = (10.0, 10.0, 20.0, 20.0), (0.0, 0.0, 40.0, 40.0)
print(f'  小框完全被大框包住:  IoU = {iou(small, big):.4f}   IoS = {ios(small, big):.4f}')
assert abs(ios(small, big) - 1.0) < 1e-12 and iou(small, big) < 0.07
print('  -> **IoU 看不见「包含」关系，IoS 能**')

In [ ]:
# ---- 用上一节的两个半截框做对比 ----
fb = [f[0] for f in frags]
fs = [f[1] for f in frags]
print(f'碎片 A = {tuple(round(t,1) for t in fb[0])}   score = {fs[0]:.3f}')
print(f'碎片 B = {tuple(round(t,1) for t in fb[1])}   score = {fs[1]:.3f}')
print(f'\nIoU(A,B) = {iou(fb[0], fb[1]):.4f}   <- NMS 用它  ->  判定为「两个不同目标」❌')
print(f'IoS(A,B) = {ios(fb[0], fb[1]):.4f}   <- NMM 用它  ->  判定为「同一目标的两块」✅')

keep = nms(fb, fs, thr=0.60)
print(f'\nNMS(IoU > 0.60) 保留 {len(keep)} 个框  ->  重复输出，且两个都是**半截框**')
merged = nmm(fb, fs, ios_thr=0.50)
mb = merged[0]['box']
print(f'NMM(IoS >= 0.50) 输出 {len(merged)} 个框: {tuple(round(t,1) for t in mb)}')
print(f'  与真值 {TGT} 的 IoU = {iou(mb, TGT):.4f}')
assert len(keep) == 2 and len(merged) == 1
assert iou(mb, TGT) > 0.999, 'NMM 的并集应当精确恢复真值框'
print('\n✅ 「同一物理实体被切成多份分别观测」的场景，需要的是**合并算子**而不是抑制算子')

In [ ]:
def is_boundary(box_local, s, margin=2.0):
    '''框是否触到所在切片的边界 —— 触到就说明它**可能**被截断。'''
    x0, y0, x1, y1 = box_local
    return x0 <= margin or y0 <= margin or x1 >= s - margin or y1 >= s - margin

print(f"{'切片':>5s} {'局部框 (x0,y0,x1,y1)':>34s}  贴边?")
for i, ((x0, y0, _, _), ds) in enumerate(zip(sl_bad, lenient)):
    for dd in ds:
        bl = tuple(round(t, 1) for t in dd['box_local'])
        print(f'{i:>5d} {str(bl):>34s}  {"是 ⚠️ 可能被截断" if is_boundary(dd["box_local"], 640) else "否"}')

# ---- 为什么 NMM 必须以「贴边」为门控 ----
up   = (700.0, 300.0, 740.0, 396.0)     # 限速牌（含下方辅助牌区域的大框）
down = (700.0, 344.0, 740.0, 396.0)     # 下方的辅助牌，完全落在上面那个大框里
print(f'\n上下叠放的两块牌:  IoU = {iou(up, down):.3f}   IoS = {ios(up, down):.3f}')
assert ios(up, down) > 0.99 and iou(up, down) < 0.6
print('  -> 无条件 NMM 会把它们**错误地并成一个大框**（IoS = 1.0）')
print('  -> 而它们都不贴边（假设它们落在某片内部），贴边门控会跳过它们  ✅')
assert not is_boundary((up[0] - 640, up[1], up[2] - 640, up[3]), 640)
assert not is_boundary((down[0] - 640, down[1], down[2] - 640, down[3]), 640)
print('\n✅ 工程配方：① 只对**贴边框**做 IoS-NMM 缝合  ② 再对全部框做一次常规 IoU-NMS 去重')
print('   ③ 贴边但配不上对的框，保留并打 truncated 标记（让跟踪与距离估计降权），而不是直接丢')

## 5 · 代价账：切片数、像素吞吐与车端延迟预算

In [ ]:
def tile_stats(W, H, s, r):
    sl, st = make_slices(W, H, s, r)
    return len(sl), len(sl) * s * s

# ---- 渐近律：N·s² ≈ W·H/(1-r)²，**几乎与 s 无关** ----
W2, H2, r2 = 4096, 3072, 0.2
asym = W2 * H2 / (1 - r2) ** 2
print(f'渐近律验证（{W2}×{H2}, r={r2}）：W·H/(1-r)² = {asym/1e6:.2f} Mpx')
print(f"\n{'s':>6s} {'N':>5s} {'实际像素吞吐(Mpx)':>18s} {'相对渐近值':>11s}")
for s_c in [512, 640, 1024]:
    n_, px_ = tile_stats(W2, H2, s_c, r2)
    dev = px_ / asym - 1
    print(f'{s_c:>6d} {n_:>5d} {px_/1e6:>18.2f} {dev:>+11.1%}')
    assert abs(dev) < 0.12, (s_c, dev)
print('\n✅ **总计算量几乎与切片大小 s 无关，只由重叠率 r 决定**（偏差来自贴边补片的离散化）')
print('   s 决定的不是成本，而是**放大倍数 W/s** —— 先按放大倍数定 s，再按下界定 r，成本自动落定')

In [ ]:
MS_PER_MPX = 5.0 / (640 * 640 / 1e6)      # 640² 一次前向 5.0 ms（Orin 级、FP16、small 量级模型）
print(f'耗时模型: {MS_PER_MPX:.2f} ms/Mpx   (640×640 一次 = 5.0 ms)')

PLANS = [('全图下采样 640×384', 1, 640 * 384),
         ('全图原生 1920×1088', 1, 1920 * 1088)]
for r_ in [0.0, 0.2, 0.5]:
    n_, px_ = tile_stats(W, H, 640, r_)
    PLANS.append((f'切片 s=640 r={r_:.1f}', n_, px_))
n_, px_ = tile_stats(W, H, 512, 0.2)
PLANS.append(('切片 s=512 r=0.2', n_, px_))

base = 640 * 384 * MS_PER_MPX / 1e6
print(f"\n{'方案':<22s} {'前向次数':>8s} {'像素(Mpx)':>10s} {'耗时(ms)':>9s} {'相对全图':>9s}")
for name, n_, px_ in PLANS:
    ms = px_ * MS_PER_MPX / 1e6
    print(f'{name:<22s} {n_:>8d} {px_/1e6:>10.2f} {ms:>9.1f} {ms/base:>8.1f}×')

TSR_BUDGET_MS = 8.0
ms_slice = tile_stats(W, H, 640, 0.2)[1] * MS_PER_MPX / 1e6
print(f'\n30 FPS -> 整帧 {1000/30:.1f} ms，要装下 ISP、多相机、障碍物、车道、红绿灯、TSR、融合、预测、规控')
print(f'TSR 这一路真实能分到的通常是 5–10 ms（本课按 {TSR_BUDGET_MS:.0f} ms 算）')
print(f'切片 (s=640, r=0.2) 需要 {ms_slice:.1f} ms = 预算的 {ms_slice/TSR_BUDGET_MS:.1f} 倍   ->   ❌ 车端不可行')
assert ms_slice / TSR_BUDGET_MS >= 4.9
print('\n✅ 「切片能提升小目标」必须连着「代价是 4–16×、车端装不下」一起说 —— 这是面试的分水岭')
print('   切片在车端仍成立的两个用法：① 离线自动标注/教师模型  ② 低频触发的单 ROI 高分辨率精检')

## 6 · 两级级联 vs 切片 vs 全图：端到端延迟与召回

级联的召回是**乘法**：`R = R_level1 × R_level2`。
Level 1 漏掉的，Level 2 永远看不到 —— 所以 Level 1 必须极度偏向召回。

In [ ]:
def det_recall(px, p50, k=0.5):
    '''检出概率随目标在**网络输入里**的像素尺寸的 logistic 曲线（合成模型，用于相对比较）。'''
    return 1.0 / (1.0 + math.exp(-k * (px - p50)))

P50_DET, P50_PROP = 12.0, 5.0     # 完整检测（要判类别） vs 类别无关 + 低阈值的提议
Z_FAR = 60.0
p_native = F_PX * S_SIGN / Z_FAR
print(f'{Z_FAR:.0f} m 外的 {S_SIGN*100:.0f} cm 标志：原生 {p_native:.2f} px')
print(f'  -> 全图 letterbox 到 640 宽:  {p_native*640/W:.2f} px   （丢掉了 8/9 的像素）')
print(f'  -> 切片 / ROI 原生分辨率:     {p_native:.2f} px')

SCHEMES = {}
SCHEMES['全图下采样 640×384'] = dict(lat=3.0, pxin=p_native * 640 / W,
                                     recall=det_recall(p_native * 640 / W, P50_DET))
SCHEMES['两级级联(全图 L1)'] = dict(lat=6.2, pxin=p_native,
                                    recall=det_recall(p_native * 640 / W, P50_PROP) * det_recall(p_native, P50_DET))
SCHEMES['两级级联(ROI 引导 L1)'] = dict(lat=5.0, pxin=p_native,
                                        recall=det_recall(p_native / 2, P50_PROP) * det_recall(p_native, P50_DET))
SCHEMES['SAHI 切片 s=640 r=0.2'] = dict(lat=ms_slice, pxin=p_native,
                                        recall=det_recall(p_native, P50_DET))

N_FRAMES = 5
print(f"\n{'方案':<24s} {'进网络px':>9s} {'单帧召回':>9s} {'5帧累积':>9s} {'延迟ms':>8s} {'预算内':>7s}")
for name, v in SCHEMES.items():
    v['seq'] = 1 - (1 - v['recall']) ** N_FRAMES
    print(f'{name:<24s} {v["pxin"]:>9.2f} {v["recall"]:>9.3f} {v["seq"]:>9.4f} '
          f'{v["lat"]:>8.1f} {"✅" if v["lat"] <= TSR_BUDGET_MS else "❌":>7s}')

assert SCHEMES['全图下采样 640×384']['recall'] < 0.06
assert SCHEMES['两级级联(ROI 引导 L1)']['seq'] > 0.99
assert SCHEMES['SAHI 切片 s=640 r=0.2']['seq'] > SCHEMES['两级级联(ROI 引导 L1)']['seq']
print('\n✅ 单帧召回 0.76 vs 0.91 的差距，5 帧累积后变成 0.9993 vs 0.99999 —— **几乎抹平**')
print('   而 5.0 ms vs 40.0 ms 的延迟差距，**是抹不平的**  ->  量产选级联')

In [ ]:
# ---- 时序累积曲线：多少帧才够 ----
names = list(SCHEMES)
print(f"{'帧数':>5s}" + ''.join(f'{n[:22]:>24s}' for n in names))
for nf in [1, 2, 3, 5, 8, 12]:
    row = f'{nf:>5d}'
    for n in names:
        row += f'{1-(1-SCHEMES[n]["recall"])**nf:>24.4f}'
    print(row)

M_PER_FRAME = 100 / 3.6 / 30      # 100 km/h，30 FPS
print(f'\n自车 100 km/h、30 FPS -> 每帧前进 {M_PER_FRAME:.2f} m')
need = {}
for n in names:
    k = 1
    while 1 - (1 - SCHEMES[n]['recall']) ** k < 0.95 and k < 999:
        k += 1
    need[n] = k
    print(f'  {n:<24s} 达到 95% 累积召回需 {k:>3d} 帧 = {k*M_PER_FRAME:>6.1f} m 的接近距离')
assert need['全图下采样 640×384'] > 10 * need['两级级联(ROI 引导 L1)']
print('\n⚠️  独立性假设是**乐观的**：遮挡、逆光、运动模糊会让相邻帧的失败强相关，实际需要更多帧。')
print('    正确的做法是在真实序列上直接测「首次检出距离」的分布，而不是用单帧召回外推。')

## ✏️ 练习 1：重叠率下界与切片规划

实现 `min_overlap_ratio(d_max, s)`：返回保证「任意位置、尺寸 ≤ d_max 的目标都能被某一片
**完整包含**」的最小重叠率。再实现 `plan_slices(W, H, s, d_max)`，返回
`{'overlap_ratio', 'stride', 'overlap_px', 'n_slices', 'magnification'}`
（`stride` 用 `int(round(s*(1-r)))`，`magnification = W/s`）。

In [ ]:
def min_overlap_ratio(d_max, s):
    # TODO: 由 s·r >= d_max 推出
    raise NotImplementedError

def plan_slices(W, H, s, d_max):
    # TODO: 取下界重叠率，算出 stride / overlap_px / 片数 / 放大倍数
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(min_overlap_ratio(64, 640) - 0.10) < 1e-12
assert abs(min_overlap_ratio(96, 480) - 0.20) < 1e-12
assert min_overlap_ratio(700, 640) > 1.0, 'd_max > s 时无解（几何上不可能）'

p = plan_slices(1920, 1080, 640, 64)
print(p)
assert p['stride'] == 576 and p['overlap_px'] == 64 and p['n_slices'] == 8
assert abs(p['magnification'] - 3.0) < 1e-9
assert uncovered_positions(1920, 640, p['overlap_ratio'], 64)[0] == []

print(f"\n{'d_max':>7s} {'r 下界':>8s} {'stride':>7s} {'N':>4s} {'放大':>6s} {'零失败?':>8s}")
for dm in [32, 64, 96, 128]:
    pp = plan_slices(1920, 1080, 640, dm)
    bad = uncovered_positions(1920, 640, pp['overlap_ratio'], dm)[0]
    print(f'{dm:>7d} {pp["overlap_ratio"]:>8.4f} {pp["stride"]:>7d} {pp["n_slices"]:>4d} '
          f'{pp["magnification"]:>5.1f}× {"✅" if not bad else "❌":>8s}')
    assert bad == []
print('\n✅ 练习 1 通过：**overlap_px >= d_max** 是切片推理唯一必须记住的不等式')

## ✏️ 练习 2：完整的跨片合并流水线

实现 `merge_slice_detections(per_slice_dets, slices, s, ios_thr=0.5, iou_thr=0.6, margin=2.0)`：

1. **坐标还原**：局部框 + 切片原点 → 全图坐标；同时用 `is_boundary` 打**贴边标记**
2. **只对贴边框**做 `nmm`（IoS 合并），缝合被截断的碎片；未被合并的（`members` 只有 1 个）
   打上 `truncated=True`
3. 把「合并后的贴边框」与「非贴边框」放在一起做一次 `nms`（IoU 去重）
4. 返回 `[{'box':…, 'score':…, 'truncated':…}, …]`，按分数降序

In [ ]:
def merge_slice_detections(per_slice_dets, slices, s, ios_thr=0.5, iou_thr=0.6, margin=2.0):
    # TODO: 还原坐标 -> 贴边分流 -> NMM 缝合 -> NMS 去重
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# ① 低于下界产生的两个半截框，应当被缝合成一个精确的框
out_frag = merge_slice_detections(lenient, sl_bad, 640)
print('碎片场景 ->', [(tuple(round(t,1) for t in o['box']), round(o['score'],3), o['truncated'])
                      for o in out_frag])
assert len(out_frag) == 1
assert iou(out_frag[0]['box'], TGT) > 0.999
assert out_frag[0]['truncated'] is False, '被成功缝合的框不应再标记为截断'

# ② 满足下界的完整场景：12 个目标 -> 12 个框，且一一对应
out_full = merge_slice_detections(per_slice, slices, 640)
print(f'\n完整场景: 还原前 {sum(len(d) for d in per_slice)} 个框  ->  合并后 {len(out_full)} 个框'
      f'（真值 {len(GT)} 个）')
assert len(out_full) == len(GT)
matched = set()
for o in out_full:
    best = int(np.argmax([iou(o['box'], tuple(g)) for g in GT]))
    assert iou(o['box'], tuple(GT[best])) > 0.99, o
    matched.add(best)
assert matched == set(range(len(GT))), '每个真值目标应当恰好对应一个输出框'
print('✅ 练习 2 通过：**贴边门控 + IoS-NMM + IoU-NMS** 是跨片合并的标准三件套')

## ✏️ 练习 3：延迟预算下的切片配置规划

实现 `slice_latency_ms(W, H, s, r, ms_per_mpx=MS_PER_MPX)` 与
`plan_under_budget(W, H, d_max, budget_ms, s_candidates, ms_per_mpx=MS_PER_MPX)`：

对每个候选切片边长 s（跳过 `d_max >= s` 的），取**下界重叠率** `r = d_max/s`，算出延迟；
在预算内返回**放大倍数最高（即 s 最小）**的那个配置 dict
（含 `'s' / 'r' / 'n_slices' / 'latency_ms' / 'magnification'`）；都不可行返回 `None`。

In [ ]:
def slice_latency_ms(W, H, s, r, ms_per_mpx=MS_PER_MPX):
    # TODO
    raise NotImplementedError

def plan_under_budget(W, H, d_max, budget_ms, s_candidates, ms_per_mpx=MS_PER_MPX):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
CAND = [320, 448, 640, 896]
print(f"{'s':>6s} {'r 下界':>8s} {'N':>4s} {'延迟(ms)':>10s} {'放大':>6s}")
for s_c in CAND:
    r_c = 64 / s_c
    print(f'{s_c:>6d} {r_c:>8.4f} {tile_stats(1920,1080,s_c,r_c)[0]:>4d} '
          f'{slice_latency_ms(1920,1080,s_c,r_c):>10.2f} {1920/s_c:>5.1f}×')

assert abs(slice_latency_ms(1920, 1080, 640, 0.2) - 40.0) < 1e-6

r45 = plan_under_budget(1920, 1080, 64, 45.0, CAND)
assert r45 is not None and r45['s'] == 320, r45
assert abs(r45['magnification'] - 6.0) < 1e-9

r38 = plan_under_budget(1920, 1080, 64, 38.0, CAND)
assert r38 is not None and r38['s'] == 448, r38

assert plan_under_budget(1920, 1080, 64, 20.0, CAND) is None, '20 ms 预算下没有可行切片配置'
print(f'\n预算 45 ms -> {r45}')
print(f'预算 38 ms -> {r38}')
print('预算 20 ms -> None（**这正是车端的真实处境**）')
print('\n✅ 练习 3 通过：注意 s=896 反而更贵 —— 渐近律在 s 与 H 同量级时失效（贴边补片的冗余）')

## ✏️ 练习 4：给定延迟预算的方案决策器

实现 `choose_strategy(budget_ms, schemes, n_frames=5, seq_target=0.95)`，返回 `(方案名, 方案dict)`：

1. 只考虑 `lat <= budget_ms` 的方案；一个都没有 → 返回 `(None, None)`
2. 其中 `n_frames` 帧**累积召回** `1-(1-recall)^n_frames >= seq_target` 的，取**延迟最小**的
3. 若没有达标的，取预算内**累积召回最高**的（保底方案）

返回的 dict 里要额外带上 `'seq'` 字段。

In [ ]:
def choose_strategy(budget_ms, schemes, n_frames=5, seq_target=0.95):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
for b in [2.0, 4.0, 8.0, 100.0]:
    name, info = choose_strategy(b, SCHEMES)
    if name is None:
        print(f'预算 {b:>6.1f} ms  ->  ❌ 无可行方案')
    else:
        print(f'预算 {b:>6.1f} ms  ->  {name:<24s} (延迟 {info["lat"]:.1f} ms, 5帧累积 {info["seq"]:.4f})')

assert choose_strategy(2.0, SCHEMES)[0] is None
assert choose_strategy(4.0, SCHEMES)[0] == '全图下采样 640×384', '预算太紧时只能退到保底方案'
assert choose_strategy(8.0, SCHEMES)[0] == '两级级联(ROI 引导 L1)'
assert choose_strategy(100.0, SCHEMES)[0] == '两级级联(ROI 引导 L1)', \
    '即使预算无限，达标方案里也该选**最省**的那个，而不是召回最高的'
print('\n✅ 练习 4 通过：**达标之后就不要再花钱买召回** ——')
print('   把省下的 35 ms 给别的感知任务，比把 TSR 单帧召回从 0.76 推到 0.91 有价值得多')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def min_overlap_ratio(d_max, s):
    return d_max / s                      # 由 s·r >= d_max 直接得到

def plan_slices(W, H, s, d_max):
    r = min_overlap_ratio(d_max, s)
    stride = max(1, int(round(s * (1.0 - r))))
    n = len(slice_origins(W, s, stride)) * len(slice_origins(H, s, stride))
    return {'overlap_ratio': r, 'stride': stride, 'overlap_px': s - stride,
            'n_slices': n, 'magnification': W / s}

In [ ]:
# 练习 2 参考答案
def merge_slice_detections(per_slice_dets, slices, s, ios_thr=0.5, iou_thr=0.6, margin=2.0):
    edge, inner = [], []
    for (x0, y0, _, _), ds in zip(slices, per_slice_dets):
        for d in ds:
            bx0, by0, bx1, by1 = d['box_local']
            rec = {'box': (bx0 + x0, by0 + y0, bx1 + x0, by1 + y0), 'score': float(d['score'])}
            (edge if is_boundary(d['box_local'], s, margin) else inner).append(rec)

    cand = [{'box': r['box'], 'score': r['score'], 'truncated': False} for r in inner]
    if edge:                                        # ① 只对贴边框做 IoS 合并
        for m in nmm([e['box'] for e in edge], [e['score'] for e in edge], ios_thr):
            cand.append({'box': m['box'], 'score': m['score'],
                         'truncated': len(m['members']) == 1})   # 配不上对 -> 保留但打标记
    if not cand:
        return []
    keep = nms([c['box'] for c in cand], [c['score'] for c in cand], iou_thr)  # ② 常规 NMS 去重
    return sorted([cand[i] for i in keep], key=lambda c: -c['score'])

In [ ]:
# 练习 3 参考答案
def slice_latency_ms(W, H, s, r, ms_per_mpx=MS_PER_MPX):
    return tile_stats(W, H, s, r)[1] * ms_per_mpx / 1e6

def plan_under_budget(W, H, d_max, budget_ms, s_candidates, ms_per_mpx=MS_PER_MPX):
    for s in sorted(s_candidates):                  # s 从小到大 = 放大倍数从高到低
        if d_max >= s:
            continue                                # 目标比切片还大，几何上无解
        r = min_overlap_ratio(d_max, s)
        ms = slice_latency_ms(W, H, s, r, ms_per_mpx)
        if ms <= budget_ms:
            return {'s': s, 'r': r, 'n_slices': tile_stats(W, H, s, r)[0],
                    'latency_ms': ms, 'magnification': W / s}
    return None

In [ ]:
# 练习 4 参考答案
def choose_strategy(budget_ms, schemes, n_frames=5, seq_target=0.95):
    aff = {k: v for k, v in schemes.items() if v['lat'] <= budget_ms}
    if not aff:
        return None, None
    seq = {k: 1 - (1 - v['recall']) ** n_frames for k, v in aff.items()}
    ok = [k for k in aff if seq[k] >= seq_target]
    key = min(ok, key=lambda k: aff[k]['lat']) if ok else max(aff, key=lambda k: seq[k])
    return key, dict(aff[key], seq=seq[key])

---
## 🧪 真实工程胶囊：一份可直接用的切片/级联配置与检查清单

In [ ]:
RECIPE = r'''
# ============ ① 决定要不要切片：先算「目标进网络时还剩几个像素」 ============
f_px      = (W_img / 2) / tan(radians(HFOV) / 2)       # 60°/1920 -> 1662.77
p_native  = f_px * S_object / Z_target                 # 0.6 m 标志 @ 60 m -> 16.63 px
p_input   = p_native * (L_net / L_crop)                # 全图 letterbox 640 -> 5.54 px
#   p_input >= 32 px  -> 别切，没收益
#   p_input <  12 px  -> 切片/级联能带来两位数 AP_small 提升

# ============ ② SAHI 切片参数（离线/云端评测、自动标注用） ============
from sahi.predict import get_sliced_prediction
d_max = np.percentile(gt_sizes, 99)                    # **用 p99 而不是均值**
result = get_sliced_prediction(
    image, detection_model,
    slice_height=640, slice_width=640,                 # = 训练输入尺寸，不要乱改
    overlap_height_ratio=max(0.2, d_max / 640),        # **下界 r >= d_max/s，再往上推到片数跳变前**
    overlap_width_ratio =max(0.2, d_max / 640),
    postprocess_type="NMM",                            # **不是 NMS**：跨片碎片要合并不是抑制
    postprocess_match_metric="IOS",                    # **不是 IOU**：要度量「包含」而非「相似」
    postprocess_match_threshold=0.5,
    perform_standard_pred=True,                        # 额外跑一遍全图：接住大目标 + 补上下文
)
# 别忘了另一半：slicing aided **fine-tuning** —— 训练集也按同样 (s, r) 切，
# 与整图按 ~1:1 混合训练，否则尺度分布错位，收益会大打折扣。

# ============ ③ 车端：两级级联（量产的现实选择） ============
LEVEL1 = dict(input=(384, 640), class_agnostic=True, score_thr=0.05, topk=6)  # 高召回，精度不重要
LEVEL2 = dict(crop=256, source="**原生分辨率**", pad_ratio=0.35, score_thr=0.40)
#  纪律 A：Level 2 的训练 crop 必须由 **Level 1 的真实提议** 裁出（带定位噪声），不能用 GT 裁
#  纪律 B：ROI 先验（消失点/车道/地图）只能是**软的** —— 必须保留一路全图下采样兜底
#  纪律 C：ROI 的垂直位置用**实时估计的地平线**，不要用标定常数（上下坡会整段漏检）

# ============ ④ 上线前必查（切片/级联特有，常规检查之外） ============
CHECKS = [
  "重叠像素 s*r >= d_max(p99)？",
  "跨片合并用的是 IoS-NMM 而不是 IoU-NMS？贴边门控开了吗？",
  "letterbox 逆变换与切片原点的顺序对吗？（框整体偏移的头号原因）",
  "**按位置分桶评测**：切片边界附近 20 px 带内的召回 vs 中心区域，差多少？",
  "延迟的 p99 而不是均值；长时间跑之后有没有因为降频而漂移？",
  "Level 2 的训练分布 = Level 1 的推理输出分布吗？",
]
'''
print(RECIPE)
for k in ['p_input', 'd_max / 640', 'NMM', 'IOS', 'perform_standard_pred',
          'fine-tuning', 'class_agnostic', '真实提议', '地平线']:
    assert k in RECIPE, k
print('✅ 配方覆盖：切不切的判据 / 重叠下界 / NMM+IoS / 切片微调 / 两级级联三条纪律 / 上线检查')

### 小结

- **切片没有创造信息，只是拒绝丢弃信息**。判断值不值得切，先算
  `p_input = p_native × L_net/L_crop`：还剩 32 px 就别切，只剩 5 px 才有大收益。
- **重叠率下界 `s·r ≥ d_max`**（等价 `r ≥ d_max/s`）。推导只有三行，
  面试要能当场写出来。用 **p99 而不是均值**定 d_max，否则失败会集中在几条固定竖线上。
- 片数是 r 的**阶梯函数** —— 把 r 从下界推到片数跳变前，是**免费的鲁棒性**。
- **跨片合并要用 NMM + IoS，而不是 NMS + IoU**：IoU 度量「两框有多像」，
  IoS 度量「小框有多大比例被包住」。并且必须加**贴边门控**，
  否则上下叠放的限速牌+辅助牌会被并成一个大框。
- **代价的渐近律 `N·s² ≈ W·H/(1-r)²`**：总算力几乎与切片大小无关，只由重叠率决定。
  s 决定的是**放大倍数**而不是成本。
- **切片在车端量产系统里基本不可用**（40 ms vs 8 ms 预算）。
  能说清这一点并给出替代方案，比会背 SAHI 高一个层次。
- **两级级联是现实选择**：召回是乘法 `R1×R2`，所以 Level 1 必须类别无关 + 低阈值 + 极度偏召回。
  单帧召回 0.76 vs 切片 0.91，**5 帧累积后变成 0.9993 vs 0.99999，差距被时序抹平；
  而 8 倍的延迟差抹不平。**
- 三条路径共享同一条纪律：**推理时输入被什么变换裁剪/缩放过，训练时就必须复现那个变换**
  （切片 → SF 微调；级联 → 用真实提议裁 crop；ROI → 把 ROI 分布放进训练集）。

下一站：**模块 05 · TSR 小目标实战** —— 从针孔模型 `px = f·S/Z` 出发，
把「要在多远检出」反推成「需要什么分辨率、什么 stride、什么相机」。